In [ ]:
!pip install jupyter-dash

In [3]:
!pip install dash dash-leaflet plotly pandas numpy matplotlib

In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

username = "aacuser"
password = "DD214Apr18"

db = AnimalShelter(username, password)

test_data = db.read({})
print("Records loaded:", len(test_data))

df = pd.DataFrame.from_records(test_data)

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

print(df.head())
print(df.columns)


Records loaded: 10000
     age_upon_outcome animal_id animal_type                    breed  \
0  1          3 years   A746874         Cat   Domestic Shorthair Mix   
1  2           1 year   A725717         Cat   Domestic Shorthair Mix   
2  3          2 years   A716330         Dog  Chihuahua Shorthair Mix   
3  4         7 months   A733653         Cat              Siamese Mix   
4  5          2 years   A691584         Dog   Labrador Retriever Mix   

          color date_of_birth             datetime            monthyear  \
0   Black/White    2014-04-10  2017-04-11 09:00:00  2017-04-11T09:00:00   
1  Silver Tabby    2015-05-02  2016-05-06 10:49:00  2016-05-06T10:49:00   
2   Brown/White    2013-11-18  2015-12-28 18:43:00  2015-12-28T18:43:00   
3    Seal Point    2016-01-25  2016-08-27 18:11:00  2016-08-27T18:11:00   
4     Tan/White    2012-11-06  2015-05-30 13:48:00  2015-05-30T13:48:00   

  outcome_subtype     outcome_type sex_upon_outcome  location_lat  \
0            SCRP        

In [2]:

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

 
# Radio button filer allows the client to filter dogs by rescue training type and 
# update all dashboard components
 
html.Img(
    src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '100px'}
        ),
  
app.layout = html.Div([
# html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Dashboard- Rolesshania Jackson'))),
    html.Hr(),
    html.Div(
        [
            html.H4("Filter by Rescue Type"),
            dcc.Dropdown(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue', 'value': 'Water'},
                    {'label': 'Mountain ', 'value': 'Mountain'},
                    {'label': 'Disaster ', 'value': 'Disaster'},
                    {'label': 'Reset', 'value': 'Reset'}
                ],
                value='Reset',
                clearable=False,
                placeholder='Select a Rescue Type'
            )
        ] 
        
# Interactive filer options for rescue type selection

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         row_selectable='single',
                         selected_rows=[0], 
                         style_table={'overflowX':'auto'},
                         sort_action="native",
                         page_current=0, 
                         page_size=10,
                         filter_action="native",  
    
    
    
#If you completed the Module Six Assignment, you can copy in the code you created here 

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
#Update the data table base on the rescue type selected by userd.
# This acts as the controller, while gathering filtered data from MongoDB and returning in full display.
    if filter_type== 'All':
        df = pd.Dataframe.from_records(shelter.read({}))

    if filter_type == "Water":
        data = db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix",
                              "Chesapeake Bay Retriever",
                              "Newfoundland"]},
            "sex_upon_outcome": "Intact Female"
        })

    elif filter_type == "Mountain or Wilderness Rescue":
        data = db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd",
                              "Alaskan Malamute",
                              "Old English Sheepdog"]},
            "sex_upon_outcome": "Intact Male"
        })

    elif filter_type == "Disaster or Individual Tracking":
        data = db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher",
                              "German Shepherd",
                              "Rottweiler"]},
            "sex_upon_outcome": "Intact Male"
        })

    else:
        data = db.read({})  # Reset (unfiltered)

    df_filtered = pd.DataFrame.from_records(data)

    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)

    return df_filtered.to_dict('records')

# Update the the number of graphs that will display cetain  labels
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return html.Div("No data available")

    dff = pd.DataFrame.from_dict(viewData)

    if 'age_upon_outcome_in_weeks' not in dff.columns:
        return html.Div("Age data not available")

    # Scatter Plot
    scatter_fig = px.scatter(
        dff,
        x='animal_type',
        y='age_upon_outcome_in_weeks',
        color='breed',
        title='Age Distribution of Rescue Dogs',
        labels={'age_upon_outcome_in_weeks': 'Age (Weeks)'}
    )

    return [
        dcc.Graph(figure=scatter_fig)
    ]

    
#This callback will highlight a cell on the data table when the user selects it
# DASH triggers this callback on page load before column is selected, so selected_columns can be None
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # If no columns are selected, return an empty list
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]



# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


app.run(mode="inline")